<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/18_fine_tuning_an_llm_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Fine-Tuning an LLM

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

On Day 4 we got an LLM to code bill titles by asking it in different ways: zero-shot, with a
codebook, with examples. The model itself never changed. In this notebook it does: we **fine-tune**
a small open LLM on human-coded titles, so the task is built into the model instead of spelled out
in every prompt.

Fine-tuning all of a model's weights takes a lot of memory. We use **LoRA**, which freezes the model
and trains a small add-on of less than 1% of its size. That fits on Colab's free GPU in a few minutes,
and the result is a file of a few megabytes you can save, share, and switch on or off.

> ⚠️ **Turn on the GPU first:** *Runtime → Change runtime type → T4 GPU.*

In this notebook you will:

- decide when fine-tuning an LLM is worth it, compared with prompting, BERT or distillation
- understand LoRA: what is trained, and why it is cheap
- turn coded examples into training conversations, and train only on the answers
- fine-tune Qwen2.5 0.5B on bill titles and compare it with the same model prompted with a codebook
- check what else the fine-tuning changed, and save, reload and share the adapter

**Contents**

1. When to fine-tune an LLM
2. LoRA in one picture
3. The data
4. The base model, prompted
5. Training data: conversations with masked prompts
6. Fine-tuning with LoRA
7. Did it work?
8. What else changed?
9. Saving and sharing the adapter
10. Wrap-up

There are 5 exercises along the way, marked ✏️. The solutions notebook has worked answers, but try them yourself first.

## 0. Setup

In [ ]:
!pip install -q transformers accelerate datasets peft
!pip uninstall -y -q torchao      # Colab's preinstalled torchao is too old for current PEFT; we don't need it

import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import hf_hub_download
from peft import LoraConfig, PeftModel, get_peft_model
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from transformers import (AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq, Trainer,
                          TrainingArguments, set_seed)

SEED = 42
set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 120)
print("GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "none: Runtime → Change runtime type → T4 GPU")

---
## 1. When to fine-tune an LLM

| | What changes | Labeled data | Cost per text | Good for |
|---|---|---|---|---|
| **Prompting** (Day 4) | the prompt | none, plus a test set | long prompt every time | trying things, small jobs |
| **Few-shot** (Day 4) | a few examples in the prompt | a handful | even longer prompts | when the boundaries need showing |
| **Fine-tuning BERT** (Day 3) | a small encoder | hundreds | very low | classification at scale |
| **Distillation** (Day 4) | a small encoder, trained on LLM labels | LLM labels, plus a test set | very low | scale when human labels are scarce |
| **Fine-tuning an LLM** (here) | a small add-on to the LLM | hundreds | short prompt | consistent formats, generation tasks, a task too nuanced for a prompt |

For pure classification, a fine-tuned BERT is often just as good and much cheaper to run. Fine-tuning an
LLM pays off when the output is **text**: extraction into a fixed format (Bonus 15), coding with
explanations, rewriting in a house style. Here we use classification anyway, because the Day 3 and
Day 4 results on the same titles give us something to compare with.

One thing fine-tuning doesn't do well: teaching the model **new facts**. It changes how the model
behaves, much more than what it knows.

---
## 2. LoRA in one picture

A layer in the model multiplies its input by a large weight matrix $W$ (say 896 × 896). Full
fine-tuning updates all of $W$. **LoRA** (Hu et al., 2022) keeps $W$ frozen and adds a correction made of
two thin matrices:

$$W' = W + \frac{\alpha}{r} \, B A, \qquad A: r \times 896, \quad B: 896 \times r$$

With rank $r = 16$, the two thin matrices have about 29,000 numbers instead of 800,000. Only they are
trained, so the optimizer needs a fraction of the memory, and the result is a small **adapter** file.
**QLoRA** (Dettmers et al., 2023) goes further and stores the frozen model in 4 bits, which brings 7B
models within reach of a single Colab GPU.

---
## 3. The data

The same bill titles and the same four policy areas as on Day 3 and Day 4, with the same development
(10 per area) and test (25 per area) samples as the Day 4 notebook. From the remaining pool we draw a
**training set** of 200 titles per area.

In [ ]:
parts = []
for chunk in [0, 5]:          # two of the ten Parquet files, as on Day 3
    path = hf_hub_download(repo_id="dreamproit/bill_labels_us", repo_type="dataset",
                           revision="refs/convert/parquet", filename=f"default/train/{chunk:04d}.parquet")
    parts.append(pd.read_parquet(path, columns=["title", "policy_area"]))
raw = pd.concat(parts, ignore_index=True)

LABELS = ["Health", "Education", "Taxation", "Armed Forces and National Security"]

bills = (raw.assign(title=raw["title"].fillna("").str.strip())
            .query("title != ''")
            .drop_duplicates(subset="title"))
bills = bills[bills["policy_area"].isin(LABELS)].sample(frac=1, random_state=SEED).reset_index(drop=True)

dev = bills.groupby("policy_area").head(10)
rest = bills.drop(dev.index)
test = rest.groupby("policy_area").head(25)
pool = rest.drop(test.index)
train = pool.groupby("policy_area").sample(200, random_state=SEED).sample(frac=1, random_state=SEED)
dev, test, train = (d.reset_index(drop=True) for d in (dev, test, train))
print(f"train: {len(train)}   development: {len(dev)}   test: {len(test)}")

---
## 4. The base model, prompted

We fine-tune **Qwen2.5 0.5B Instruct**: a third of the size of the Day 4 model, so training takes
minutes. First a baseline: the same small model, prompted with the Day 4 codebook.

In [ ]:
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
base_model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device)

CODEBOOK = """You code US bill titles by policy area.

Categories:
- Health: health care and health insurance, Medicare and Medicaid, diseases and public health, drugs and medical devices, medical research.
- Education: schools and teachers, colleges and universities, student aid and student loans, education programs.
- Taxation: taxes of any kind, the Internal Revenue Code, tax credits and deductions, the IRS.
- Armed Forces and National Security: the military and defense, servicemembers and veterans, intelligence.
If a bill fits several categories, choose the one that matches its main purpose.

Reply with the category name only."""
SHORT = "You code US bill titles by policy area. Reply with the category name only."


def messages(title, system):
    return [{"role": "system", "content": system}, {"role": "user", "content": f"Bill title: {title}"}]


@torch.no_grad()
def answer(model, msgs, max_new_tokens=12):
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                         max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def to_label(text):
    found = [lab for lab in LABELS if lab.lower() in text.lower()]
    if not found and ("armed forces" in text.lower() or "national security" in text.lower()):
        found = ["Armed Forces and National Security"]
    return found[0] if len(found) == 1 else "INVALID"


def evaluate(model, df, system, name, build=messages):
    start = time.time()
    labels = pd.Series([to_label(answer(model, build(t, system))) for t in df["title"]], index=df.index)
    prompt_tokens = len(tokenizer(tokenizer.apply_chat_template(build(df["title"][0], system), tokenize=False,
                                                                add_generation_prompt=True))["input_ids"])
    row = {"strategy": name, "accuracy": accuracy_score(df["policy_area"], labels),
           "macro-F1": f1_score(df["policy_area"], labels, labels=LABELS, average="macro", zero_division=0),
           "invalid": (labels == "INVALID").mean(), "prompt tokens": prompt_tokens,
           "seconds per title": (time.time() - start) / len(df)}
    print(f"{name}: accuracy {row['accuracy']:.2f}, macro-F1 {row['macro-F1']:.2f}, invalid {row['invalid']:.0%}")
    return row, labels


results = []
row, base_labels = evaluate(base_model, test, CODEBOOK, "0.5B, codebook prompt")
results.append(row)

---
## 5. Training data: conversations with masked prompts

Each coded title becomes a short conversation: the instruction, the title, and the correct answer from
the assistant. Because the knowledge will sit in the model, the instruction can be **short**: no
codebook needed.

The model learns by predicting the next token, but we only want it to learn the **answer**, not to
reproduce the instruction and the title. So every prompt token gets the label `-100`, which tells the
training loop to ignore it. Only the answer tokens, and the end-of-turn token that teaches the model
to stop, count in the loss.

In [ ]:
def encode(title, label, system=SHORT):
    prompt = tokenizer.apply_chat_template(messages(title, system), tokenize=False, add_generation_prompt=True)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(label + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    return {"input_ids": prompt_ids + answer_ids,
            "attention_mask": [1] * (len(prompt_ids) + len(answer_ids)),
            "labels": [-100] * len(prompt_ids) + answer_ids}


def to_dataset(df):
    return Dataset.from_list([encode(t, y) for t, y in zip(df["title"], df["policy_area"])])


train_ds = to_dataset(train)
example = train_ds[0]
print(tokenizer.decode(example["input_ids"]))
print("Tokens in the example:", len(example["input_ids"]), "· of which in the loss:",
      sum(l != -100 for l in example["labels"]))

> **✏️ Exercise 1**
>
> Decode only the tokens of `example` that count in the loss (where the label isn't `-100`), and then the
> tokens that don't. What would the model learn if we didn't mask the prompt?

In [ ]:
# Your code here

---
## 6. Fine-tuning with LoRA

We add LoRA adapters to the attention layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`) with rank 16, and
train for two passes over the 800 titles. The helper below starts from a fresh copy of the base model
each time, so the exercises can train their own variants.

In [ ]:
def train_lora(train_df, r=16, epochs=2, learning_rate=2e-4, batch_size=8, seed=SEED):
    set_seed(seed)
    model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device)
    config = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, task_type="CAUSAL_LM",
                        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
    model = get_peft_model(model, config)
    model.print_trainable_parameters()
    steps = math.ceil(len(train_df) / batch_size) * epochs
    args = TrainingArguments(output_dir="lora-run", num_train_epochs=epochs, learning_rate=learning_rate,
                             per_device_train_batch_size=batch_size, warmup_steps=int(0.1 * steps),
                             lr_scheduler_type="cosine", logging_steps=max(1, steps // 10),
                             save_strategy="no", report_to="none", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=to_dataset(train_df),
                      data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
    start = time.time()
    trainer.train()
    print(f"Training took {(time.time() - start) / 60:.1f} minutes")
    losses = pd.DataFrame(trainer.state.log_history).dropna(subset=["loss"])[["step", "loss"]]
    return model.eval(), losses


ft_model, losses = train_lora(train)
losses.plot(x="step", y="loss", marker="o", legend=False, figsize=(6, 3), ylabel="training loss")
plt.tight_layout()
plt.show()

"Trainable params" shows the point of LoRA: well under 1% of the model is trained. The loss should drop
fast: the answers are short and come from a list of four.

---
## 7. Did it work?

The fine-tuned model gets only the short instruction, no codebook, on the same 100 test titles.

In [ ]:
row, ft_labels = evaluate(ft_model, test, SHORT, "0.5B, fine-tuned (LoRA), short prompt")
results.append(row)
pd.DataFrame(results).set_index("strategy").round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(test["policy_area"], ft_labels, labels=LABELS,
                                        display_labels=[l.replace(" and ", "\n& ") for l in LABELS],
                                        cmap="Greens", colorbar=False, ax=ax)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

Things to look at:

- **Accuracy.** Fine-tuned smaller models regularly beat much larger models used zero-shot on text
  classification (Bucher & Martini, 2024). Compare with your Day 4 results (Qwen2.5 1.5B, five
  strategies) and the Day 3 fine-tuned DistilBERT on the same test titles.
- **Invalid answers.** After fine-tuning, the model answers in exactly the trained format.
- **Prompt tokens.** The fine-tuned model needs no codebook in the prompt, so every title costs fewer
  tokens. Over a large corpus that adds up.

> **✏️ Exercise 2**
>
> **How much data do you need?** Train fresh adapters on 25 and 100 titles per area (sample them from
> `train` with `groupby("policy_area").sample(...)`), evaluate each on the **development** set, and plot
> accuracy against the number of training titles, together with the model trained on all 800. Free GPU
> memory with `del` and `torch.cuda.empty_cache()` between runs.

In [ ]:
# Your code here

> **✏️ Exercise 3**
>
> **Fine-tuning or just examples?** Give the base model eight coded examples in the prompt instead (two
> per area from `train`, as earlier turns of the conversation, like few-shot on Day 4), and evaluate it on
> the test set. Compare accuracy, prompt tokens and seconds per title with the fine-tuned model.

In [ ]:
# Your code here

---
## 8. What else changed?

We trained on one narrow task. What does the model do now with anything else? PEFT lets us switch the
adapter off temporarily, so we can ask the same question to the model with and without it.

In [ ]:
question = [{"role": "user", "content": "In one sentence, what does a member of parliament do?"}]
print("With the adapter:   ", answer(ft_model, question, max_new_tokens=40))
with ft_model.disable_adapter():
    print("Without the adapter:", answer(ft_model, question, max_new_tokens=40))

A model fine-tuned on one task often keeps doing that task, even when asked something else: it has
become a specialist. For a research instrument that is fine, as long as you only use it for what it was
trained on. Because the adapter is separate from the base model, you can keep several adapters for
different tasks and switch between them. LoRA also tends to forget less of what the base model could do
than full fine-tuning (Biderman et al., 2024).

> **✏️ Exercise 4**
>
> Write three general prompts of your own (a translation, a definition, a short summary of a bill
> title) and compare the answers with and without the adapter. Then ask the fine-tuned model to code a
> **Dutch** title, for example "Wetsvoorstel over de financiering van het basisonderwijs". Does it
> generalize to a language it wasn't fine-tuned on?

In [ ]:
# Your code here

---
## 9. Saving and sharing the adapter

The adapter is all you trained, so it's all you need to save: a few megabytes instead of a gigabyte.
Anyone with the same base model can load it.

In [ ]:
ft_model.save_pretrained("bill-coder-lora")
size = sum(os.path.getsize(os.path.join("bill-coder-lora", f)) for f in os.listdir("bill-coder-lora"))
print(sorted(os.listdir("bill-coder-lora")), f"{size / 1e6:.1f} MB")

reloaded = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device),
                                     "bill-coder-lora").eval()
title = test["title"][0]
print(title, "→", answer(reloaded, messages(title, SHORT)))

- To use the model without PEFT, **merge** the adapter into the weights with `merge_and_unload()` and
  save the full model.
- To **share** it, push the adapter to the Hugging Face Hub (Day 3). Think first: a model can reveal
  parts of its training data, so don't share adapters trained on confidential or personal texts.
- **License:** the adapter inherits the base model's license. Qwen2.5 0.5B is Apache 2.0; other models
  set conditions (see the Day 4 ethics slides).

> **✏️ Exercise 5**
>
> **How big does the adapter need to be?** Train an adapter with rank `r=2` instead of 16, and compare
> the number of trainable parameters, the size of the saved adapter, and the accuracy on the development
> set.

In [ ]:
# Your code here

---
## 10. Wrap-up

- **Fine-tune an LLM when the output is text** in a fixed form, or when a long prompt would otherwise
  travel with every text. For plain classification at scale, a fine-tuned BERT or distillation is
  usually cheaper.
- **LoRA** trains a small adapter on top of a frozen model; QLoRA adds 4-bit storage for bigger models.
- **Train on the answers only**, and keep a development and a test set apart, as always.
- **Check what else changed**: a fine-tuned model is a specialist.
- **Report** the base model and version, the training data and its size, the LoRA settings (rank, alpha,
  target layers), epochs, learning rate and seed, and the test scores. Share the adapter if the data
  allows it.
- For larger jobs, libraries such as TRL and Unsloth wrap these steps and add tricks for speed and memory;
  the ideas are the same.

### Further reading

- Hu, E. J., et al. (2022). LoRA: Low-rank adaptation of large language models. *ICLR 2022*.
- Dettmers, T., Pagnoni, A., Holtzman, A., & Zettlemoyer, L. (2023). QLoRA: Efficient finetuning of
  quantized LLMs. *NeurIPS 2023*.
- Biderman, D., et al. (2024). LoRA learns less and forgets less. *Transactions on Machine Learning
  Research*.
- Bucher, M. J. J., & Martini, M. (2024). Fine-tuned "small" LLMs (still) significantly outperform
  zero-shot generative AI models in text classification. *arXiv:2406.08660*.
- Hugging Face PEFT documentation: [huggingface.co/docs/peft](https://huggingface.co/docs/peft)